# Capstone Project 2
## Retail Demand Forecasting & Inventory Optimization


**Objective:**  
Develop a machine learning model to forecast daily retail sales and design an inventory reorder strategy to minimize stock-outs and overstock risks.

**Dataset:**  
Rossmann Store Sales dataset (Kaggle)

**Tools Used:**  
Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn

In [ ]:
# ================================
# 1. Import Required Libraries
# ================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

# Ignore warnings

import warnings
warnings.filterwarnings("ignore")

# Set display options
pd.set_option("display.max_columns", None)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ================================
# 2. Load and Merge Dataset
# ================================

train = pd.read_csv("/content/drive/MyDrive/train.csv")
store = pd.read_csv("/content/drive/MyDrive/store.csv")

# Merge sales data with store metadata
df = train.merge(store, on="Store", how="left")

print("Dataset Shape:", df.shape)
df.head()

The dataset contains daily sales transactions merged with store-level attributes such as store type, competition distance, and promotional details.

# 3. Data Understanding

This section explores the structure, variables, and data quality before preprocessing.

In [ ]:
# ================================
# 3.1 Dataset Shape
# ================================

print("Number of Rows:", df.shape[0])
print("Number of Columns:", df.shape[1])

In [ ]:
# ================================
# 3.2 Dataset Information
# ================================

df.info()

The dataset contains both numerical and categorical variables.
The target variable for forecasting is **Sales**.

In [ ]:
# ================================
# 3.3 Statistical Summary
# ================================

df.describe()

In [ ]:
# ================================
# 3.4 Missing Values
# ================================

df.isnull().sum().sort_values(ascending=False)

Certain competition and promotion-related columns contain missing values, which will be handled in the data cleaning stage.

In [ ]:
# ================================
# 3.5 Sales Distribution
# ================================

plt.figure()
df["Sales"].hist(bins=50)
plt.title("Sales Distribution")
plt.xlabel("Sales")
plt.ylabel("Frequency")
plt.show()

The sales distribution appears right-skewed, indicating the presence of high-value outliers.
This suggests potential need for outlier treatment.

# 4. Data Cleaning

This section prepares the dataset for modeling by handling invalid records and missing values.

In [ ]:
# ================================
# 4.1 Remove Closed Stores & Zero Sales
# ================================

# Remove days when store was closed
df = df[df["Open"] == 1]

# Remove zero sales entries
df = df[df["Sales"] > 0]

print("Shape after removing closed stores and zero sales:", df.shape)

Closed store days and zero-sales records were removed to ensure accurate demand modeling.

In [ ]:
# ================================
# 4.2 Convert Date Column
# ================================

df["Date"] = pd.to_datetime(df["Date"])

The Date column was converted to datetime format to enable time-based feature extraction.

In [ ]:
# ================================
# 4.3 Handle Missing Values
# ================================

# Fill CompetitionDistance with median
df["CompetitionDistance"].fillna(df["CompetitionDistance"].median(), inplace=True)

# Fill competition and promo related columns with 0
cols_to_fill_zero = [
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear",
    "Promo2SinceWeek",
    "Promo2SinceYear"
]

for col in cols_to_fill_zero:
    df[col].fillna(0, inplace=True)

# Check remaining missing values
df.isnull().sum().sort_values(ascending=False).head()


Missing values in competition and promotional variables were treated using median imputation (for distance) and zero imputation (for time-based promo indicators).

### Handling Categorical Missing Values

The `PromoInterval` column contains missing values for stores that do not participate in the continuous promotion program (Promo2).  
These missing values are replaced with `"None"` to preserve categorical consistency and accurately represent non-participating stores.

In [ ]:
# ================================
# 4.4 Handle PromoInterval Missing Values
# ================================

df["PromoInterval"].fillna("None", inplace=True)

# Verify missing values
df.isnull().sum().sort_values(ascending=False).head()

# 5. Feature Engineering

Feature engineering is performed to capture seasonality, weekly demand patterns, and historical sales behavior to improve forecasting accuracy.

## 5.1 Time-Based Feature Engineering

To capture seasonal and temporal sales patterns, additional features are extracted from the `Date` column.

These features help the model understand:
- Yearly trends
- Monthly seasonality
- Weekly demand cycles
- Weekend vs weekday behavior
- Quarterly business performance patterns

In [ ]:
# ================================
# 5.1 Time-Based Features
# ================================

df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Week"] = df["Date"].dt.isocalendar().week
df["DayOfWeek"] = df["Date"].dt.dayofweek
df["Quarter"] = df["Date"].dt.quarter

# Weekend indicator
df["IsWeekend"] = df["DayOfWeek"].apply(lambda x: 1 if x >= 5 else 0)

df.head()

## 5.2 Lag and Rolling Features

Lag and rolling features are created to help the model learn historical demand patterns and short-term trends.

In [ ]:
# ================================
# 5.2 Lag & Rolling Features
# ================================

# Sort by Store and Date for correct time-series calculations
df = df.sort_values(["Store", "Date"])

# Sales 7 days ago
df["Lag_7"] = df.groupby("Store")["Sales"].shift(7)

# Rolling average of last 7 days
df["Rolling_Mean_7"] = (
    df.groupby("Store")["Sales"]
    .shift(1)
    .rolling(7)
    .mean()
)

# Drop rows with NA values created by lag features
df = df.dropna()

print("Shape after feature engineering:", df.shape)

# 6. Exploratory Data Analysis (EDA)

This section explores sales trends, seasonality, and promotional impact to derive business insights before model building.

## 6.1 Monthly Sales Trend

Analyzing average monthly sales to identify seasonality patterns.

In [ ]:
# ================================
# 6.1 Monthly Sales Trend
# ================================

monthly_sales = df.groupby("Month")["Sales"].mean()

plt.figure()
monthly_sales.plot()
plt.title("Average Monthly Sales")
plt.xlabel("Month")
plt.ylabel("Average Sales")
plt.show()


### Interpretation

Sales exhibit seasonal variation across months.  
Certain months demonstrate higher average demand, indicating strong seasonality effects.

## 6.2 Promotion Impact on Sales

Evaluating the effect of promotional campaigns on sales performance.

In [ ]:
# ================================
# 6.2 Promotion Impact on Sales
# ================================
promo_sales = df.groupby("Promo")["Sales"].mean()

plt.figure()
promo_sales.plot(kind="bar")
plt.title("Average Sales: Promo vs No Promo")
plt.xlabel("Promo (0 = No, 1 = Yes)")
plt.ylabel("Average Sales")
plt.show()

### Interpretation

Sales during promotional periods are significantly higher than non-promotional days.  
This highlights the strong impact of promotional strategies on revenue generation.

## 6.3 Day-of-Week Sales Pattern

Analyzing weekly demand variations.

In [ ]:
# ================================
# 6.3 Day-of-Week Sales Pattern
# ================================

weekly_sales = df.groupby("DayOfWeek")["Sales"].mean()

plt.figure()
weekly_sales.plot()
plt.title("Average Sales by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Average Sales")
plt.show()

### Interpretation

Sales vary across weekdays, indicating cyclical weekly demand behavior.  
Understanding this pattern helps improve short-term forecasting accuracy.

## 6.4 Sales by Store Type

Analyzing how different store categories perform in terms of average sales.

In [ ]:
# ================================
# 6.4 Sales by Store Type
# ================================

store_type_sales = df.groupby("StoreType")["Sales"].mean().sort_values(ascending=False)

plt.figure()
store_type_sales.plot(kind="bar")
plt.title("Average Sales by Store Type")
plt.xlabel("Store Type")
plt.ylabel("Average Sales")
plt.show()

### Interpretation

Average sales vary across store types, indicating structural differences in scale and customer base.  
Certain store categories consistently outperform others in revenue generation.

## 6.5 Competition Distance vs Sales

Analyzing whether proximity to competitors impacts sales performance.

In [ ]:
# ================================
# 6.5 Competition Distance vs Sales
# ================================

plt.figure()
plt.scatter(df["CompetitionDistance"], df["Sales"])
plt.title("Competition Distance vs Sales")
plt.xlabel("Competition Distance")
plt.ylabel("Sales")
plt.show()

### Interpretation

There is no strong linear relationship between competition distance and sales.  
While nearby competitors may affect demand, other factors such as promotions and store type play a more significant role.

# 7. Model Building

This section develops multiple regression models to forecast daily sales.
A progressive modeling approach is used, starting from a simple linear model to advanced ensemble techniques.

## 7.1 Data Preparation

Preparing feature matrix (X) and target variable (y) for model training.

In [ ]:
# ================================
# 7.1 Encode Categorical Variables
# ================================

# Convert categorical variables into dummy variables
df_model = pd.get_dummies(df, drop_first=True)

df_model.shape

In [ ]:
# ================================
# 7.2 Define Features and Target
# ================================

X = df_model.drop(["Sales", "Date", "Customers"], axis=1)
y = df_model["Sales"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

## 7.2 Train-Test Split

Splitting data into training and testing sets to evaluate model performance.

In [ ]:
# ================================
# 7.3 Train-Test Split
# ================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

# 8. Model Development

This section develops multiple regression models to forecast daily retail sales.
A progressive modeling approach is used, starting from a baseline linear model to advanced ensemble techniques.

## 8.1 Linear Regression (Baseline Model)

A simple linear regression model is built to establish a performance benchmark.

In [ ]:
# ================================
# Linear Regression
# ================================

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

## 8.2 Decision Tree Regressor

A decision tree model is used to capture non-linear relationships in the data.

In [ ]:
# ================================
# Decision Tree
# ================================

dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

## 8.3 Random Forest Regressor

A bagging-based ensemble model built using multiple decision trees to improve stability and performance.

In [ ]:
# ================================
# Random Forest
# ================================

rf = RandomForestRegressor(
    n_estimators=50,
    max_depth=18,
    min_samples_split=10,
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

## 8.4 XGBoost Regressor

XGBoost is a boosting-based ensemble model that sequentially improves prediction errors and is known for strong performance on structured tabular data.

In [ ]:
from xgboost import XGBRegressor

# ================================
# XGBoost Regressor
# ================================

xgb = XGBRegressor(
    n_estimators=80,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

# 9. Model Evaluation

The models are evaluated using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and R² Score to compare predictive performance.

In [ ]:
# ================================
# 9.1 Evaluation Function
# ================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

In [ ]:
# ================================
# 9.2 Model Performance Comparison
# ================================

lr_metrics = evaluate_model(y_test, y_pred_lr)
dt_metrics = evaluate_model(y_test, y_pred_dt)
rf_metrics = evaluate_model(y_test, y_pred_rf)
xgb_metrics = evaluate_model(y_test, y_pred_xgb)

print("Linear Regression:", lr_metrics)
print("Decision Tree:", dt_metrics)
print("Random Forest:", rf_metrics)
print("XGBoost:", xgb_metrics)

In [ ]:
# ================================
# 9.3 Comparison Table
# ================================

comparison_df = pd.DataFrame({
    "Model": ["Linear Regression", "Decision Tree", "Random Forest", "XGBoost"],
    "MAE": [lr_metrics[0], dt_metrics[0], rf_metrics[0], xgb_metrics[0]],
    "RMSE": [lr_metrics[1], dt_metrics[1], rf_metrics[1], xgb_metrics[1]],
    "R2 Score": [lr_metrics[2], dt_metrics[2], rf_metrics[2], xgb_metrics[2]]
})

comparison_df

### Feature Selection Consideration

The `Customers` variable was excluded from the modeling process to prevent data leakage and simulate realistic forecasting conditions.

Since future customer count is unknown at the time of prediction, the model was trained using only historical sales and business-related features.

## 9.4 Final Model Selection

Based on evaluation metrics, the Random Forest Regressor demonstrated the best predictive performance, achieving the lowest MAE and RMSE along with the highest R² score.

### Hyperparameter Consideration

Hyperparameter tuning was considered; however, the baseline Random Forest model already achieved strong performance (R² ≈ 0.90).
Further tuning was avoided to prevent overfitting and reduce computational cost.

# 10. Feature Importance Analysis

Feature importance is analyzed using the final selected model (Random Forest) to understand which factors most influence sales performance.

In [ ]:
# ================================
# 10.1 Feature Importance (Random Forest)
# ================================

import pandas as pd

feature_importances = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf.feature_importances_
})

feature_importances = feature_importances.sort_values(
    by="Importance", ascending=False
)

feature_importances.head(10)

In [ ]:
# ================================
# 10.2 Plot Top 10 Features
# ================================

top_features = feature_importances.head(10)

plt.figure()
plt.barh(top_features["Feature"], top_features["Importance"])
plt.gca().invert_yaxis()
plt.title("Top 10 Feature Importances (Random Forest)")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.show()

### Feature Importance Interpretation

Rolling 7-day average sales emerged as the most influential predictor, highlighting strong short-term demand momentum.

Promotional activity and weekly seasonality (DayOfWeek and Week) significantly impact sales performance.

Lag-based features further confirm temporal dependence in retail demand, indicating that recent historical patterns are critical for accurate forecasting.

# 11. Inventory Optimization

Using forecasted demand, an inventory reorder strategy is designed to minimize stock-outs and excess holding costs.

## 11.1 Reorder Point Formula

Inventory decisions are based on the following logic:

Reorder Point = (Average Daily Demand × Lead Time) + Safety Stock

Safety Stock = Z × Demand Standard Deviation × √Lead Time

Where:
- Lead Time = Number of days required for replenishment
- Z = Service level factor (1.65 for 95% service level)
- Demand Std Dev = Variability in historical demand

In [ ]:
# ================================
# 11.2 Inventory Reorder Calculation
# ================================

# Calculate average demand per store
avg_demand = df.groupby("Store")["Sales"].mean()

# Calculate demand standard deviation per store
std_demand = df.groupby("Store")["Sales"].std()

# Assumptions
lead_time = 7  # days
z_value = 1.65  # 95% service level

# Safety Stock
safety_stock = z_value * std_demand * np.sqrt(lead_time)

# Reorder Point
reorder_point = (avg_demand * lead_time) + safety_stock

# Create summary dataframe
inventory_df = pd.DataFrame({
    "Average_Daily_Demand": avg_demand,
    "Demand_StdDev": std_demand,
    "Safety_Stock": safety_stock,
    "Reorder_Point": reorder_point
})

inventory_df.head()

### Inventory Strategy Interpretation

Stores with higher average demand and higher variability require larger safety stock levels.

For example, Store 4 has high average demand and volatility, resulting in a significantly higher reorder point compared to smaller stores.

This approach ensures stock availability during lead time while minimizing the risk of overstocking.

# 12. Business Impact & Conclusion

This project developed a retail demand forecasting system using machine learning and business analytics.

Key Achievements:
- Built multiple regression models and selected Random Forest as the best-performing approach.
- Prevented data leakage by excluding customer count from forecasting.
- Identified key sales drivers including rolling demand trends and promotional activity.
- Designed an inventory reorder strategy using safety stock principles.

Business Value:
- Improved demand visibility
- Reduced stock-out risk
- Optimized inventory levels
- Enabled data-driven decision making

This end-to-end approach integrates SQL-based historical analysis with predictive modeling and operational inventory planning.

In [ ]:
# =================================
# 13. Model Deployment Preparation
# =================================

# 13.1 Save Final Trained Model (Random Forest)

import pickle

with open("rf_model.pkl", "wb") as f:
    pickle.dump(rf, f)

print("Random Forest model saved successfully.")

In [ ]:
# =================================
# 13.2 Save Feature Column Order
# =================================

with open("model_columns.pkl", "wb") as f:
    pickle.dump(X_train.columns.tolist(), f)

print("Feature columns saved successfully.")

In [ ]:
# =================================
# 13.3 Download Deployment Files
# =================================

from google.colab import files

files.download("rf_model.pkl")
files.download("model_columns.pkl")

print("Deployment files downloaded.")